In [42]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_1.csv')
# print(df.info())
# print(df.head())
df1=df.groupby('소주제')['정답여부'].value_counts().reset_index()
df2 = df1.pivot(index='소주제', columns='정답여부', values='count')
df2
df2['정답률']=df2[1] / (df2[0] + df2[1])
df3=df2['정답률'].sort_values(ascending=False)
df3
df3.rank(method='dense', ascending=False)
print(df3['경제'])

0.68


In [132]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_2.csv')
# print(df.info())
# print(df.head())
df['연도']=df['date'].str.split('-').str[0]
df['월']=df['date'].str.split('-').str[1]
# df.info()
print(df.groupby(['연도', '월'])['price'].sum().sort_values(ascending=False)[1])
# 2024 12
print(df[(df['연도'] == '2024') & (df['월'] == '12')].groupby('category')['price'].sum().max())

1777389
1012500


C:\Users\baekj\AppData\Local\Temp\ipykernel_17244\3571783513.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df.groupby(['연도', '월'])['price'].sum().sort_values(ascending=False)[1])


In [133]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_3.csv')
# print(df.info())
# print(df.head())
def count(x):
    return len(x)
df['count'] = df['message'].str.split(' ').apply(count)
print(np.abs(df.groupby('label')['count'].mean()[0]-df.groupby('label')['count'].mean()[1]).round(3))

0.3


C:\Users\baekj\AppData\Local\Temp\ipykernel_17244\3370641262.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(np.abs(df.groupby('label')['count'].mean()[0]-df.groupby('label')['count'].mean()[1]).round(3))
C:\Users\baekj\AppData\Local\Temp\ipykernel_17244\3370641262.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(np.abs(df.groupby('label')['count'].mean()[0]-df.groupby('label')['count'].mean()[1]).round(3))


In [154]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_test.csv')
print(train_df.info())
print(train_df.head())
print(test_df.info())

train_df=train_df[train_df['gas_totl']!=0]

train_y=train_df['gas_totl']
train_X=train_df.drop(columns=['gas_totl'])
test_y=test_df['gas_totl']
test_X=test_df.drop(columns=['gas_totl'])

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import get_scorer_names, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
# neg_root_mean_squared_error

train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.2)
print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()

encoder=OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_X_encoder=encoder.fit_transform(train_X[obj])
valid_X_encoder=encoder.transform(valid_X[obj])
test_X_encoder=encoder.transform(test_X[obj])

train_X=np.concatenate([train_X_encoder, train_X[num]],axis=1)
valid_X=np.concatenate([valid_X_encoder, valid_X[num]],axis=1)
test_X=np.concatenate([test_X_encoder, test_X[num]],axis=1)

rf=RandomForestRegressor()
rf.fit(train_X, train_y)
pred1=rf.predict(valid_X)
print(root_mean_squared_error(valid_y,pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring=root_mean_squared_error, cv=3)
cv.fit(train_X, train_y)
pred2=cv.predict(valid_X)
print(root_mean_squared_error(valid_y,pred2))
pred3=cv.predict(test_X)

pd.DataFrame(pred3,columns=['pred']).to_csv('result.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   biz_type        160 non-null    object 
 1   area            160 non-null    float64
 2   age             160 non-null    int64  
 3   num_households  160 non-null    int64  
 4   gas_totl        160 non-null    int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 6.4+ KB
None
      biz_type       area  age  num_households  gas_totl
0       public  93.291780   33              61       499
1  residential  97.509495   34             131      1156
2  residential  97.140181    5              30       392
3  residential  65.789408   12             166      1343
4        mixed  92.631000   28              14       154
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          ----

C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 194, in wrapper
    params = func_sig.bind(*args, **kwargs)
  File "C:\Users\baekj\anaconda3\Lib\inspect.py", line 3295, in bind
    return self._bind(args, kwargs)
           ~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\baekj\anaconda3\Lib\inspect.py", line 3219, in _bind
    raise TypeError(
        'too many positional arguments') from None
TypeError: too many positional arguments

  warnings.warn(
C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:960: Us

74.09763877409834


C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py", line 194, in wrapper
    params = func_sig.bind(*args, **kwargs)
  File "C:\Users\baekj\anaconda3\Lib\inspect.py", line 3295, in bind
    return self._bind(args, kwargs)
           ~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\baekj\anaconda3\Lib\inspect.py", line 3219, in _bind
    raise TypeError(
        'too many positional arguments') from None
TypeError: too many positional arguments

  warnings.warn(
C:\Users\baekj\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserW

In [138]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_1.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.glm("attrition ~ age + income + overtime", data=df, family=sm.families.Binomial()).fit()
# model.pvalues[1:] < 0.05
print(model.params['income'].round(3))
print(np.round(np.exp(model.params['age']),3))
test_X=pd.DataFrame({'age': [20], 'income': [3000], 'overtime': [2]})
pred1=model.predict(test_X)
pred1
print(pred1[0].round(3))

-0.005
0.905
0.431


In [128]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_2.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.ols("price ~ area + height + wall", data=df).fit()
model.pvalues[1:] < 0.05
rs=model.params[1:]
print(np.round(rs['area'] + rs['height'], 3))

model1=smf.ols("price ~ area + height", data=df).fit()
print(np.round(model1.rsquared,3))

test_X=pd.DataFrame({'area': [100], 'height': [10], 'wall': [1]})
pred1=model1.predict(test_X)
print(np.round(pred1[0],3))

10.289
0.859
329.036
